# Dựng tập dev — 500 câu warmup

Tập dev KHÔNG phải một lát cắt của train. `warmup.json` là 500 câu độc lập, và ba khác
biệt có chủ đích so với tập train (xem docstring `clean_data.ghi_warmup`):

1. `answer` giữ **nguyên văn** — leaderboard chấm với reference thô của BTC.
2. Không lọc `is_usable_for_sft`, không khử trùng — bỏ mẫu khó khỏi dev là tự nâng điểm.
3. **Giữ cả câu retrieval trượt gold** (`--giu-het`). Tập train đã loại hẳn nhóm này, nên
   nếu dev cũng loại thì không còn chỗ nào đo được model bịa khi context thiếu.

`build_sft.py` vẫn ghi cột `ly_do_loai` cho từng dòng, nên lúc train tách được METEOR của
nhóm trúng gold và nhóm trượt (`meteor_truot` trong `train_sft.py`).

~500 câu × 1,80 s = **~15 phút GPU**.

Cần gắn: T4 x2, Internet ON, `official` + `legalqa-v10-src` + `legalqa-warmup`,
model `qwen3-5-2b-base`, và kernel output `v10-sft-budget` (có sẵn `clean/`).

In [ ]:
import os, sys, shutil, zipfile
from pathlib import Path

SRC = None
for cand in Path("/kaggle/input").glob("**/*"):
    if cand.name == "legalqa" and (cand / "bm25.py").exists():
        SRC = cand.parent.parent
        break
if SRC is None:
    z = next(Path("/kaggle/input").glob("**/src.zip"), None)
    assert z, "Không thấy codebase trong /kaggle/input — chưa gắn dataset legalqa-v10-src"
    with zipfile.ZipFile(z) as f:
        f.extractall("/kaggle/working/unz")
    SRC = Path("/kaggle/working/unz")

REPO = Path("/kaggle/working/repo")
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(SRC, REPO)

!pip -q install transformers==4.44.2 sentencepiece

os.chdir(REPO)
sys.path.insert(0, "src")

import torch
print("codebase:", (REPO / "COMMIT.txt").read_text().strip() if (REPO / "COMMIT.txt").exists() else "KHÔNG có COMMIT.txt")
print("GPU     :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ — bật Accelerator")

## CLEAN ghép bằng symlink, không copy

`clean/` của lần chạy trước nằm dưới `/kaggle/input` — **read-only**, mà `build_sft` cần
ghi được (cache `bm25.pkl`). Copy thì tốn 1,2 GB và vài phút cho `chunks.jsonl` +
`corpus.jsonl`; symlink cho ra một thư mục ghi được mà không nhân bản byte nào.

Hai file dev (`warmup_sft.jsonl`, `gold_warmup.jsonl`) dựng sẵn trên máy dev và đi vào
bằng dataset `legalqa-warmup` — dựng lại trên Kaggle đòi chạy lại `build_gold.py`, tức
quét lại 287k Chunk, 26 phút quota GPU cho một file 96 KB.

In [ ]:
cands = [p.parent for p in Path("/kaggle/input").glob("**/selected-contexts") if p.is_dir()]
assert cands, "Không thấy selected-contexts/ trong /kaggle/input — chưa gắn dữ liệu thi"
DATA_DIR = cands[0]

cu = next((p for p in Path("/kaggle/input").glob("**/clean") if (p / "chunks.jsonl").exists()), None)
assert cu, "Không thấy clean/ của lần chạy trước — gắn kernel output v10-sft-budget"

CLEAN = Path("/kaggle/working/clean")
CLEAN.mkdir(parents=True, exist_ok=True)
for f in cu.iterdir():
    d = CLEAN / f.name
    if not d.exists():
        d.symlink_to(f)

w = next((p.parent for p in Path("/kaggle/input").glob("**/warmup_sft.jsonl")), None)
assert w, "Không thấy warmup_sft.jsonl — dataset legalqa-warmup chưa lên bản mới"
for n in ("warmup_sft.jsonl", "gold_warmup.jsonl"):
    shutil.copy(w / n, CLEAN / n)

os.environ["DATA_DIR"] = str(DATA_DIR)
os.environ["CLEAN_DIR"] = str(CLEAN)
os.environ["PYTHONPATH"] = "src"

print("CLEAN", CLEAN)
for f in sorted(CLEAN.iterdir()):
    print(f"  {f.name:22} {'->' if f.is_symlink() else '  '} {f.stat().st_size/1e6:8.1f} MB")

In [ ]:
import json

local = [p.parent for p in Path("/kaggle/input").glob("**/config.json")
         if "XLMRobertaForSequenceClassification" in p.read_text()]
if local:
    MODEL = str(local[0])
    print("dùng bản đã gắn:", MODEL)
else:
    from huggingface_hub import snapshot_download
    MODEL = snapshot_download("AITeamVN/Vietnamese_Reranker")
    print("tải từ HuggingFace:", MODEL)

tk = next((p for p in Path("/kaggle/input").glob("**/tokenizer.json") if "qwen" in str(p).lower()), None)
assert tk, "Không thấy tokenizer Qwen — gắn model qwen3-5-2b-base"
os.environ["QWEN_TOKENIZER"] = str(tk)
print("tokenizer:", tk)

## Sinh dev

`--giu-het` ghi cả dòng bị loại kèm `ly_do_loai`. Ba số cần nhìn: tổng 500, số
`retrieval_truot_gold` (so với 11,9% của train), và số `vuot_max_seq` — nếu nhóm cuối
lớn hơn train đáng kể thì `MAX_SEQ=6144` chốt trên train đang hẹp so với phân phối thật.

In [ ]:
!python scripts/build_sft.py --model $MODEL --batch-size 32 --pool 50 \
    --nguon warmup_sft.jsonl --gold gold_warmup.jsonl --giu-het \
    --ra /kaggle/working/dev.jsonl

In [ ]:
from collections import Counter
rows = [json.loads(l) for l in open("/kaggle/working/dev.jsonl", encoding="utf-8")]
print("tổng", len(rows))
for k, v in Counter(r["ly_do_loai"] or "GIỮ" for r in rows).most_common():
    print(f"  {k:22} {v:4}  {100*v/len(rows):5.1f}%")
sl = sorted(r["seq_len"] for r in rows)
print(f"seq_len  p50 {sl[len(sl)//2]}  p90 {sl[int(.9*len(sl))]}  max {sl[-1]}")